# Greedy Algorithms — LeetCode Questions (8 Qs)

Greedy = make locally optimal choice at each step, hoping for global optimum.
Key: prove greedy works (exchange argument or proof by contradiction).

When greedy works: Interval scheduling, jump games, Huffman coding.
When it DOESN'T: coin change (some coin systems), shortest path (need Dijkstra/DP).

1. 455 – Assign Cookies
2.  55 – Jump Game
3.  45 – Jump Game II
4.  56 – Merge Intervals
5.  57 – Insert Interval
6. 435 – Non-overlapping Intervals
7. 763 – Partition Labels
8. 134 – Gas Station

In [ ]:
from typing import List

# 1. LeetCode 455 – Assign Cookies

Give each greedy child (with greed g[i]) a cookie (with size s[j] >= g[i]).
Maximize number of content children.

Input: g=[1,2,3], s=[1,1]  Output: 1

## Dry Run

Sort both. Use two pointers: try smallest cookie for least greedy child.

Sorted g=[1,2,3], s=[1,1]

| child ptr | cookie ptr | g[child] | s[cookie] | satisfied? | action |
|-----------|------------|----------|-----------|------------|--------|
| 0 | 0 | 1 | 1 | yes (1>=1) | child++, cookie++ |
| 1 | 1 | 2 | 1 | no (1<2) | cookie++ |
| 1 | 2 | | out of cookies | done | |
| Result: 1 child satisfied | | | | | |

In [ ]:
class Solution:
    def findContentChildren(self, g: List[int], s: List[int]) -> int:
        g.sort()
        s.sort()
        child = cookie = 0
        while child < len(g) and cookie < len(s):
            if s[cookie] >= g[child]:
                child += 1     # child satisfied
            cookie += 1        # move to next cookie regardless
        return child

print(Solution().findContentChildren([1,2,3], [1,1]))   # 1
print(Solution().findContentChildren([1,2], [1,2,3]))   # 2

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n log n + m log m) – sorting | O(1) – two pointer variables |
| **Final: O(n log n)** | **Final: O(1)** |

# 2. LeetCode 55 – Jump Game

Each element = max jump from that position. Can you reach the last index?

Input: [2,3,1,1,4]  Output: True
Input: [3,2,1,0,4]  Output: False

## Dry Run

Track max_reach: how far we can currently reach.

nums = [2,3,1,1,4]

| i | nums[i] | max_reach | i <= max_reach? | update max_reach |
|---|---------|-----------|-----------------|-----------------|
| 0 | 2 | 0 | yes (0<=0) | max(0, 0+2)=2 |
| 1 | 3 | 2 | yes (1<=2) | max(2, 1+3)=4 |
| 2 | 1 | 4 | yes | max(4, 2+1)=4 |
| 3 | 1 | 4 | yes | max(4, 3+1)=4 |
| 4 | 4 | 4 | yes | max(4, 4+4)=8 |
| i=4 = last index, reachable → **True** | | | | |

nums = [3,2,1,0,4]: at i=3, max_reach=3, i>max_reach → **False**

In [ ]:
class Solution:
    def canJump(self, nums: List[int]) -> bool:
        max_reach = 0
        for i, jump in enumerate(nums):
            if i > max_reach:
                return False          # can't reach this position
            max_reach = max(max_reach, i + jump)
        return True

print(Solution().canJump([2,3,1,1,4]))   # True
print(Solution().canJump([3,2,1,0,4]))   # False

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n) – single pass | O(1) – one variable |
| **Final: O(n)** | **Final: O(1)** |

# 3. LeetCode 45 – Jump Game II

Minimum number of jumps to reach last index.

Input: [2,3,1,1,4]  Output: 2  (0→1→4)

Greedy: at each jump, pick the position that allows reaching farthest.

## Dry Run

nums = [2,3,1,1,4]

| jump | current range | max next reach | action |
|------|--------------|----------------|--------|
| 0 | [0,0] | max(0+2,)=2 from pos 0 | jump when needed |
| 1 | [1,2] (pos 0's range) | max(1+3,2+1)=4 | must jump to leave [0,0] |
| 2 | [3,4] | reached last | done |
| Result: **2 jumps** | | | |

In [ ]:
class Solution:
    def jump(self, nums: List[int]) -> int:
        jumps = 0
        curr_end = 0       # end of current jump's range
        farthest = 0       # farthest we can reach

        for i in range(len(nums) - 1):    # don't need to jump from last
            farthest = max(farthest, i + nums[i])
            if i == curr_end:              # must take a jump now
                jumps += 1
                curr_end = farthest

        return jumps

print(Solution().jump([2,3,1,1,4]))   # 2
print(Solution().jump([2,3,0,1,4]))   # 2

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n) – single pass | O(1) – constant variables |
| **Final: O(n)** | **Final: O(1)** |

# 4. LeetCode 56 – Merge Intervals

Merge all overlapping intervals.

Input: [[1,3],[2,6],[8,10],[15,18]]  Output: [[1,6],[8,10],[15,18]]

## Dry Run

Sort by start time. Merge when current start <= previous end.

Sorted: [[1,3],[2,6],[8,10],[15,18]]

| current | last in result | overlap? | action |
|---------|---------------|----------|--------|
| [1,3] | none | - | add [1,3] |
| [2,6] | [1,3] | 2<=3 yes | merge: [1, max(3,6)]=[1,6] |
| [8,10] | [1,6] | 8<=6 no | add [8,10] |
| [15,18] | [8,10] | 15<=10 no | add [15,18] |
| Result: [[1,6],[8,10],[15,18]] | | | |

In [ ]:
class Solution:
    def merge(self, intervals: List[List[int]]) -> List[List[int]]:
        intervals.sort(key=lambda x: x[0])   # sort by start
        result = [intervals[0]]

        for start, end in intervals[1:]:
            last_end = result[-1][1]
            if start <= last_end:             # overlap
                result[-1][1] = max(last_end, end)
            else:
                result.append([start, end])

        return result

print(Solution().merge([[1,3],[2,6],[8,10],[15,18]]))  # [[1,6],[8,10],[15,18]]
print(Solution().merge([[1,4],[4,5]]))                  # [[1,5]]

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n log n) – sort dominates | O(n) – result list |
| **Final: O(n log n)** | **Final: O(n)** |

# 5. LeetCode 57 – Insert Interval

Insert new interval into sorted non-overlapping intervals. Merge if needed.

Input: intervals=[[1,3],[6,9]], newInterval=[2,5]  Output: [[1,5],[6,9]]

## Dry Run

| interval | new=[2,5] | overlap? | action |
|----------|-----------|----------|--------|
| [1,3] | new.start=2 <= 3 | yes | merge: [min(1,2), max(3,5)]=[1,5] |
| [6,9] | new=[1,5], 6>5 | no overlap | add [1,5] then [6,9] |
| Result: [[1,5],[6,9]] | | | |

In [ ]:
class Solution:
    def insert(self, intervals: List[List[int]], newInterval: List[int]) -> List[List[int]]:
        result = []
        i = 0
        n = len(intervals)

        # Add all intervals ending before newInterval starts
        while i < n and intervals[i][1] < newInterval[0]:
            result.append(intervals[i])
            i += 1

        # Merge overlapping intervals with newInterval
        while i < n and intervals[i][0] <= newInterval[1]:
            newInterval[0] = min(newInterval[0], intervals[i][0])
            newInterval[1] = max(newInterval[1], intervals[i][1])
            i += 1
        result.append(newInterval)

        # Add remaining intervals
        while i < n:
            result.append(intervals[i])
            i += 1

        return result

print(Solution().insert([[1,3],[6,9]], [2,5]))          # [[1,5],[6,9]]
print(Solution().insert([[1,2],[3,5],[6,7],[8,10],[12,16]], [4,8]))  # [[1,2],[3,10],[12,16]]

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n) – single pass (already sorted) | O(n) – result list |
| **Final: O(n)** | **Final: O(n)** |

# 6. LeetCode 435 – Non-overlapping Intervals

Minimum number of intervals to REMOVE to make rest non-overlapping.

Input: [[1,2],[2,3],[3,4],[1,3]]  Output: 1  (remove [1,3])

Greedy: sort by end time. Keep intervals with earliest end, remove overlapping.

## Dry Run

Sort by end: [[1,2],[2,3],[1,3],[3,4]]  (sorted: [[1,2],[1,3],[2,3],[3,4]])

| interval | prev_end | overlap? | action |
|----------|----------|----------|--------|
| [1,2] | -inf | no | keep, prev_end=2 |
| [1,3] | 2 | 1<2 yes | **remove** (it ends later, greedily remove it) |
| [2,3] | 2 | 2>=2 no | keep, prev_end=3 |
| [3,4] | 3 | 3>=3 no | keep, prev_end=4 |
| Result: 1 removed | | | |

In [ ]:
class Solution:
    def eraseOverlapIntervals(self, intervals: List[List[int]]) -> int:
        intervals.sort(key=lambda x: x[1])   # sort by END time
        removed = 0
        prev_end = float('-inf')

        for start, end in intervals:
            if start < prev_end:
                removed += 1          # overlap: remove current (it ends later)
            else:
                prev_end = end        # no overlap: keep it

        return removed

print(Solution().eraseOverlapIntervals([[1,2],[2,3],[3,4],[1,3]]))  # 1
print(Solution().eraseOverlapIntervals([[1,2],[1,2],[1,2]]))        # 2

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n log n) – sort | O(1) – only prev_end variable |
| **Final: O(n log n)** | **Final: O(1)** |

# 7. LeetCode 763 – Partition Labels

Partition string so each letter appears in at most one part.
Return list of partition sizes. Maximize partitions.

Input: 'ababcbacadefegdehijhklij'  Output: [9,7,8]

## Dry Run

Step 1: For each char, find its LAST occurrence index.
Step 2: Greedy expand current partition to include last occurrence of all chars seen.

last = {a:8, b:5, c:7, d:14, e:15, f:11, g:13, h:19, i:22, j:23, k:20, l:21}

| i | char | last[char] | end | start | size added |
|---|------|-----------|-----|-------|-----------|
| 0 | a | 8 | max(0,8)=8 | | |
| 1 | b | 5 | max(8,5)=8 | | |
| ... | ... | ... | 8 | | |
| 8 | a | 8 | 8 | i==end → partition! size=8-0+1=9 | 9 |
| 9 | d | 14 | 14 | start=9 | |
| ... | | | 15 | | |
| 15 | e | 15 | i==end → partition! size=15-9+1=7 | 7 |
| ... | | | 23 | | 8 |

In [ ]:
class Solution:
    def partitionLabels(self, s: str) -> List[int]:
        # Step 1: last occurrence of each character
        last = {c: i for i, c in enumerate(s)}

        result = []
        start = end = 0

        for i, c in enumerate(s):
            end = max(end, last[c])    # extend partition to include all of char c
            if i == end:               # current position = end of partition
                result.append(end - start + 1)
                start = i + 1          # next partition starts after

        return result

print(Solution().partitionLabels('ababcbacadefegdehijhklij'))  # [9, 7, 8]
print(Solution().partitionLabels('eccbbbbdec'))                 # [10]

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n) – two passes (last occurrence + partition) | O(1) – at most 26 chars in dict |
| **Final: O(n)** | **Final: O(1)** |

# 8. LeetCode 134 – Gas Station

Circular route with gas[i] and cost[i] to travel to next station.
Find starting station to complete circuit, or -1 if impossible.

Input: gas=[1,2,3,4,5], cost=[3,4,5,1,2]  Output: 3

## Dry Run

Key insights:
1. If total_gas < total_cost → impossible (return -1)
2. If a solution exists, start from first position where cumulative tank never goes negative

| station | gas | cost | tank (cumulative) | total_surplus | action |
|---------|-----|------|-------------------|--------------|--------|
| 0 | 1 | 3 | 1-3=-2 | -2 | tank<0 → reset start=1, tank=0 |
| 1 | 2 | 4 | 0+(2-4)=-2 | -4 | tank<0 → reset start=2, tank=0 |
| 2 | 3 | 5 | 0+(3-5)=-2 | -6 | tank<0 → reset start=3, tank=0 |
| 3 | 4 | 1 | 0+(4-1)=3 | -3 | ok |
| 4 | 5 | 2 | 3+(5-2)=6 | 0 | ok |
| total_surplus=0 >= 0, start=**3** | | | | | |

In [ ]:
class Solution:
    def canCompleteCircuit(self, gas: List[int], cost: List[int]) -> int:
        total_surplus = 0
        tank = 0
        start = 0

        for i in range(len(gas)):
            net = gas[i] - cost[i]
            tank += net
            total_surplus += net
            if tank < 0:
                start = i + 1          # can't start here or before, try next
                tank = 0

        return start if total_surplus >= 0 else -1

print(Solution().canCompleteCircuit([1,2,3,4,5], [3,4,5,1,2]))  # 3
print(Solution().canCompleteCircuit([2,3,4], [3,4,3]))            # -1

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n) – single pass | O(1) – only variables |
| **Final: O(n)** | **Final: O(1)** |

> Brute Force O(n^2): try each starting station
> Greedy O(n): track running surplus, reset when negative